# Практика: DBSCAN — плотностная кластеризация

## Что вы сделаете
В этом ноутбуке вы:

1. исследуете нелинейные 2D-датасеты и убедитесь в ограничениях K-Means;
2. построите **k-distance plot** и выберете оптимальный параметр `eps`;
3. применените **DBSCAN** и проанализируете результаты;
4. сравните DBSCAN с K-Means на нелинейных данных;
5. исследуете влияние гиперпараметров `eps` и `min_samples`;
6. применените DBSCAN к реальному датасету **Wine** и проанализируете шумовые точки;
7. сравните силуэт DBSCAN и K-Means, сделаете выводы.

## Важно
- Заполняйте все ячейки с пометкой `# YOUR CODE HERE`.
- Не удаляйте проверки: они подскажут, правильно ли вы идёте.
- Вопросы для размышления помогут вам глубже понять алгоритм.

## Датасеты
- **`make_moons`** и **`make_circles`** из `sklearn.datasets`: 2D-данные с нелинейными кластерами.
- **Wine** (`load_wine`): реальный многомерный датасет (13 признаков, 3 класса).

## Что сдавать
1. Заполненный ноутбук со всеми графиками.
2. Краткие выводы в конце каждого раздела.
3. Итоговый вывод: когда DBSCAN лучше K-Means и когда нет.

## Краткая теория

### Типы точек в DBSCAN

DBSCAN задаётся двумя параметрами: радиусом окрестности `eps` (ε) и минимальным числом точек `min_samples`.

- **Корневая точка** (core): в её ε-окрестности ≥ `min_samples` точек.
- **Граничная точка** (border): в её ε-окрестности < `min_samples`, но она достижима из корневой.
- **Шумовая точка** (noise): не является ни корневой, ни граничной → метка **-1**.

### Как работает алгоритм

1. Для каждой непосещённой точки найти её ε-окрестность.
2. Если точек ≥ `min_samples` — начать новый кластер.
3. Рекурсивно расширить кластер через цепочки плотно достижимых точек.
4. Оставшиеся точки → шум (метка -1).

### Как выбрать eps: k-distance plot

Для каждой точки вычисляют расстояние до её k-го ближайшего соседа (k = `min_samples`). Отсортированный по убыванию график этих расстояний называется **k-distance plot**. Резкий перегиб («локоть») указывает на оптимальное значение `eps`.

---
## Шаг 1. Импорты и настройки

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from sklearn.cluster import DBSCAN, KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.datasets import make_moons, make_circles, load_wine
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Для воспроизводимости
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Импорты выполнены успешно!")

---
## Шаг 2. Исследование нелинейных датасетов

Сгенерируем два датасета, с которыми K-Means заведомо не справится.

In [ ]:
# Генерация датасетов
X_moons, y_moons = make_moons(n_samples=300, noise=0.08, random_state=RANDOM_STATE)
X_circles, y_circles = make_circles(n_samples=300, noise=0.05,
                                     factor=0.5, random_state=RANDOM_STATE)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(X_moons[:, 0], X_moons[:, 1],
                c=y_moons, cmap='tab10', s=20, alpha=0.8)
axes[0].set_title("make_moons (истинные метки)")
axes[0].set_xlabel("Признак 1")
axes[0].set_ylabel("Признак 2")

axes[1].scatter(X_circles[:, 0], X_circles[:, 1],
                c=y_circles, cmap='tab10', s=20, alpha=0.8)
axes[1].set_title("make_circles (истинные метки)")
axes[1].set_xlabel("Признак 1")
axes[1].set_ylabel("Признак 2")

plt.tight_layout()
plt.show()

In [ ]:
# Применим K-Means к этим датасетам
# YOUR CODE HERE
# 1. Создайте KMeans с n_clusters=2, random_state=RANDOM_STATE
# 2. Обучите на X_moons и X_circles
# 3. Получите метки кластеров
kmeans_moons = KMeans(n_clusters=2, random_state=RANDOM_STATE)

kmeans_moons.fit_predict(X_moons)

labels_km_moons = kmeans_moons.labels_

kmeans_circles = KMeans(n_clusters=2, random_state=RANDOM_STATE)
labels_km_circles = kmeans_circles.fit_predict(X_circles)

# Визуализация результатов K-Means
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(X_moons[:, 0], X_moons[:, 1],
                c=labels_km_moons, cmap='tab10', s=20, alpha=0.8)
axes[0].set_title("K-Means на make_moons (K=2)")

axes[1].scatter(X_circles[:, 0], X_circles[:, 1],
                c=labels_km_circles, cmap='tab10', s=20, alpha=0.8)
axes[1].set_title("K-Means на make_circles (K=2)")

plt.tight_layout()
plt.show()

**❓ Вопрос для размышления:** Почему K-Means не может правильно разделить «луны» и «кольца»? Связано ли это с формой функционала ошибки?

---
## Шаг 3. k-Distance Plot и выбор eps

Построим k-distance plot для датасета `make_moons`. Используем k = 5 (= `min_samples`).

In [ ]:
# Масштабируем данные
scaler_moons = StandardScaler()
X_moons_scaled = scaler_moons.fit_transform(X_moons)

k = 5

# YOUR CODE HERE
# 1. Создайте NearestNeighbors(n_neighbors=k) и обучите на X_moonsa_scaled
# 2. Вызовите .kneighbors() для получения матрицы расстояний
# 3. Возьмите расстояние до k-го соседа (последний столбец distances)
# 4. Отсортируйте по убыванию

nrst_ngbrs = NearestNeighbors(n_neighbors=k)
nrst_ngbrs.fit(X_moons_scaled)
distances, indices = nrst_ngbrs.kneighbors()

k_distances_moons = (np.sort(distances[:,-1])[::-1]) # замените

plt.figure(figsize=(8, 4))
plt.plot(k_distances_moons)
plt.xlabel("Точки (отсортированы по убыванию расстояния)")
plt.ylabel(f"Расстояние до {k}-го соседа")
plt.title("k-Distance Plot (make_moons, масштабированные)")
plt.grid(True)
plt.tight_layout()
plt.show()

eps = 0.22

print("Найдите 'локоть' на графике и запишите подходящее значение eps: ", eps)

**❓ Вопрос для размышления:** Где находится «локоть»? Как он указывает на границу между точками кластеров и шумом?
локоть лежит примерно в y0 0.22-0.25. Точки, соответствующие значениям ниже точки y0 (справа) - один кластер, точки слева помечаются как шум, ведь большие расстояния значат, что они лежат далеко от остальных точек.

---
## Шаг 4. DBSCAN на нелинейных датасетах

Применим DBSCAN к `make_moons` и `make_circles`. Обратите внимание на масштабирование.

In [ ]:
# Масштабируем make_circles
scaler_circles = StandardScaler()
X_circles_scaled = scaler_circles.fit_transform(X_circles)

# YOUR CODE HERE
# 1. Создайте DBSCAN с подобранным eps (из k-distance plot) и min_samples=5
# 2. Примените fit_predict к X_moons_scaled и X_circles_scaled
# 3. Подберите параметры так, чтобы алгоритм нашёл 2 кластера

db_moons = DBSCAN(eps=0.27, min_samples=5)   # замените
labels_db_moons = db_moons.fit_predict(X_moons_scaled)

db_circles = DBSCAN(eps=0.4, min_samples=5) 
labels_db_circles = db_circles.fit_predict(X_circles_scaled)

# Вспомогательная функция для визуализации
def plot_dbscan_results(X, labels, title, ax):
    """Визуализирует результаты DBSCAN.
    Корневые точки — крупные, граничные — обычные, шум — крестики."""
    noise_mask = labels == -1
    colors = cm.tab10(np.linspace(0, 1, max(labels)+1)) if max(labels) >= 0 else []
    
    for i in range(max(labels)+1):
        mask = labels == i
        ax.scatter(X[mask, 0], X[mask, 1], s=20, alpha=0.7,
                   color=colors[i], label=f'Кластер {i}')
    
    ax.scatter(X[noise_mask, 0], X[noise_mask, 1],
               s=30, c='black', marker='x', label='Шум (-1)')
    
    n_cl = max(labels) + 1 if max(labels) >= 0 else 0
    n_noise = noise_mask.sum()
    ax.set_title(f"{title}\nКластеров: {n_cl}, шум: {n_noise}")
    ax.legend(fontsize=8)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
plot_dbscan_results(X_moons_scaled, labels_db_moons, "DBSCAN на make_moons", axes[0])
plot_dbscan_results(X_circles_scaled, labels_db_circles, "DBSCAN на make_circles", axes[1])
plt.tight_layout()
plt.show()

---
## Шаг 5. Сравнение DBSCAN и K-Means на нелинейных данных

In [ ]:
# Сравнение силуэта DBSCAN vs K-Means для make_moons

# YOUR CODE HERE
# 1. Обучите KMeans(n_clusters=2) на X_moons_scaled
# 2. Вычислите silhouette_score для K-Means
# 3. Вычислите silhouette_score для DBSCAN (только для некластерных точек!)
#    Подсказка: исключите точки с labels == -1

kmeans_moons2 = KMeans(n_clusters=2)

kmeans_labels = kmeans_moons2.fit_predict(X_moons_scaled)  # замените

db_moons.fit(X_moons_scaled)
db_labels = db_moons.labels_

X_moons_filtered = X_moons_scaled[db_labels != -1]
db_labels_filtered = db_labels[db_labels != -1]


score_dbscan = silhouette_score(X_moons_filtered, db_labels_filtered)
score_kmeans = silhouette_score(X_moons_scaled, kmeans_labels)


print("=== Коэффициент силуэта на make_moons ===")
print(f"K-Means (K=2): {score_kmeans:.3f}" if score_kmeans is not None else "K-Means: не вычислен")
print(f"DBSCAN:        {score_dbscan:.3f}" if score_dbscan is not None else "DBSCAN: не вычислен")

**❓ Вопрос для размышления:** Какой алгоритм получил лучший силуэт? Соответствует ли это визуальному впечатлению? Можно ли всегда доверять силуэту как абсолютной мере качества?

> Лучший силуэт получил KMeans, но визуально он не так хорошо справился с задачей, как DBSCAN. Силуэту можно доверять только в линейных данных, тк он фактически измеряет плотность прилегания точек кластера к центру, в полумесяце крайние точки лежат очень далеко от центра, поэтому скор ниже.

---
## Шаг 6. Исследование влияния гиперпараметров

Посмотрим, как меняется результат DBSCAN при разных значениях `eps` и `min_samples`.

In [ ]:
# Сетка eps при фиксированном min_samples=5
eps_values = [0.05, 0.15, 0.3, 0.5, 0.8, 1.5]
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for i, eps_val in enumerate(eps_values):
    # YOUR CODE HERE
    # 1. Запустите DBSCAN(eps=eps_val, min_samples=5) на X_moons_scaled
    # 2. Получите метки
    # 3. Посчитайте число кластеров и шумовых точек
    # 4. Отобразите scatter plot с цветами по меткам
    #    Шумовые точки (label == -1) рисуйте чёрными крестиками
    
    db = DBSCAN(eps=eps_val, min_samples=5)
    labels_i = db.fit_predict(X_moons_scaled)  # замените
    
    if labels_i is not None:
        noise_mask = labels_i == -1
        
        n_cl = len(set(labels_i)) - (1 if -1 in labels_i else 0)
        n_noise = (noise_mask).sum()
        axes[i].scatter(X_moons_scaled[:, 0], X_moons_scaled[:, 1],
                        c=labels_i, cmap='tab10', s=10, alpha=0.7)
        axes[i].set_title(f"eps={eps_val}\nКластеров: {n_cl}, шум: {n_noise}")
        
        axes[i].scatter(X_moons_scaled[noise_mask, 0], X_moons_scaled[noise_mask, 1],
               s=30, c='black', marker='x', label='Шум (-1)')
    
    else:
        axes[i].set_title(f"eps={eps_val} (не вычислено)")

plt.suptitle("Влияние eps (min_samples=5, make_moons)", fontsize=13)
plt.tight_layout()
plt.show()

**❓ Вопрос для размышления:** При каком `eps` алгоритм нашёл два правильных кластера? Что происходит при слишком малом и слишком большом значении?
> при eps = 0.3, при слишком малом все точки помечаются шумом, а при слишком большом - переобучение, выделяется лишь один кластер.

---
## Шаг 7. DBSCAN на реальном датасете Wine

Теперь перейдём к реальным данным. Датасет Wine: 178 образцов вина из трёх сортов винограда, 13 химических признаков.

In [ ]:
# Загрузка датасета Wine
wine = load_wine()
X_wine = wine.data
y_wine = wine.target  # истинные метки (не используем при кластеризации!)
feature_names = wine.feature_names

print(f"Форма данных: {X_wine.shape}")
print(f"Признаки: {feature_names}")
print(f"\nСтатистика признаков:")
df_wine = pd.DataFrame(X_wine, columns=feature_names)
print(df_wine.describe().round(2))

In [ ]:
# YOUR CODE HERE
# 1. Масштабируйте X_wine с помощью StandardScaler
# 2. Постройте k-distance plot (k=4)
# 3. Визуально определите «локоть» и запишите подходящее eps
scaler = StandardScaler()
X_wine_scaled = scaler.fit_transform(X_wine)  # замените
k_wine = 4

nrst_ngbrs = NearestNeighbors(n_neighbors=k_wine)
nrst_ngbrs.fit(X_wine_scaled)
distances, indices = nrst_ngbrs.kneighbors()

k_dist_wine = (np.sort(distances[:,-1])[::-1]) # замените

plt.figure(figsize=(8, 4))
if k_dist_wine is not None:
    plt.plot(k_dist_wine)
plt.xlabel("Точки (отсортированы)")
plt.ylabel(f"Расстояние до {k_wine}-го соседа")
plt.title("k-Distance Plot (Wine, масштабированный)")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# YOUR CODE HERE
# 1. Запустите DBSCAN на X_wine_scaled с выбранным eps и min_samples=4
# 2. Выведите: число кластеров, число шумовых точек, долю шума
# 3. Если кластеров > 1, вычислите силуэт (исключив шум)

eps_wine = 2.1   # ИЗМЕНИТЕ на основе k-distance plot

db_wine = DBSCAN(eps=eps_wine, min_samples=4)

labels_wine = db_wine.fit_predict(X_wine_scaled)  # замените

print("=== Результаты DBSCAN на Wine ===")
# ваш код для вывода статистики

noise_mask = labels_wine == -1
cluster_mask = ~noise_mask

n_cl = len(set(labels_wine)) - (1 if -1 in labels_wine else 0)
n_noise = (noise_mask).sum()

if n_cl > 1:
    score = silhouette_score(X_wine_scaled[cluster_mask], labels_wine[cluster_mask])
    print(f"Силуэт (без шума): {score:.4f}")
else:
    print("Силуэт не применим (кластеров меньше 2)")

fig, axes = plt.subplots(1, 1, figsize=(14, 8))

axes.scatter(X_wine_scaled[cluster_mask, 0], X_wine_scaled[cluster_mask, 1],
                c=labels_wine[cluster_mask], cmap='tab10', s=10, alpha=0.7)
axes.scatter(X_wine_scaled[noise_mask, 0], X_wine_scaled[noise_mask, 1], s=30, c='black', marker='x', label='Шум (-1)')

axes.set_title(f"eps={eps_wine}\nКластеров: {n_cl}, шум: {n_noise}")
plt.show()

print(f"Кол-во кластеров: {n_cl}")
print(f"Кол-во шумовых точек: {n_noise}")
print(f"Доля шумовых точек: {n_noise / len(labels_wine)}")



---
## Шаг 8. Анализ шумовых точек

Шумовые точки — это объекты, которые DBSCAN не смог отнести ни к одному кластеру. В задаче кластеризации они могут нести важную информацию.

In [ ]:
# YOUR CODE HERE
# 1. Выберите строки X_wine, соответствующие шумовым точкам (labels_wine == -1)
# 2. Создайте DataFrame с признаками шумовых точек
# 3. Сравните среднее значение признаков шумовых точек
#    со средним по всей выборке (df_wine.mean())
# 4. Какие признаки у шумовых точек сильно отличаются от среднего?

noise_mask_wine = labels_wine == -1   # маска для шумовых точек
X_wine_noise = X_wine[noise_mask_wine]      # признаки шумовых точек
df_wine_noise = pd.DataFrame(data=X_wine_noise, columns=df_wine.columns)

print("=== Средние значения признаков ===")
# ваш код
print(f'=== Среднее значение шумовых точек:\n{df_wine_noise.mean()}')
print(f'=== Среднее значение общей выборки:\n{df_wine.mean()}')

**❓ Вопрос для размышления:** Чем отличаются шумовые точки от остальных? Являются ли они «плохими» данными или это реальные аномалии? 

> У шумовых вин ниже алкоголь, пролин и выше алкалин. Это реальные вина, которые просто легче, которые не вписываются в основные кластеры данного датасета.

---
## Шаг 9. Итоговое сравнение: DBSCAN vs K-Means на Wine

In [ ]:
# YOUR CODE HERE
# 1. Обучите KMeans с числом кластеров = числу кластеров DBSCAN
#    (или с n_clusters=3, так как истинных классов 3)
# 2. Вычислите силуэт для K-Means
# 3. Сравните силуэт DBSCAN и K-Means
# 4. Визуализируйте результаты обоих методов в 2D (первые два признака
#    после масштабирования или используйте PCA до 2 компонент)

# Подсказка для PCA:
from sklearn.decomposition import PCA
pca = PCA(n_components=2, random_state=RANDOM_STATE)

X_wine_2d = pca.fit_transform(X_wine_scaled)

fig, axes = plt.subplots(1, 3, figsize=(13, 5))

# DBSCAN
# ваш код
db_final = DBSCAN(eps=eps_wine, min_samples=4)
db_final_score = db_final.fit_predict(X_wine_scaled)
db_final_score_filtered = db_final_score[db_final_score != -1]
db_final_silhouette = silhouette_score(X_wine_scaled[db_final_score != -1], db_final_score_filtered)

# K-Means
# ваш код
kmeans_final = KMeans(n_clusters=3, random_state=RANDOM_STATE)
kmeans_final_score = kmeans_final.fit_predict(X_wine_scaled)
kmeans_final_silhouette = silhouette_score(X_wine_scaled, kmeans_final_score)

print("Силуэт KMeans: ", kmeans_final_silhouette)
print("Силуэт DBSCAN: ", db_final_silhouette)

axes[0].scatter(
    X_wine_2d[:,0],
    X_wine_2d[:, 1],
    c=db_final_score,
    cmap='viridis',
    s=50
)
axes[0].set_title('DBSCAN')

axes[1].scatter(
    X_wine_2d[:,0],
    X_wine_2d[:, 1],
    c=kmeans_final_score,
    cmap='viridis',
    s=50
)
axes[1].set_title('KMEANS')

axes[2].scatter(
    X_wine_2d[:,0],
    X_wine_2d[:, 1],
    c=y_wine,
    cmap='viridis',
    s=50
)
axes[2].set_title('TARGET')

plt.suptitle("DBSCAN vs K-Means на Wine (проекция PCA)", fontsize=13)
plt.tight_layout()
plt.show()

---
## Шаг 10. Итоговые выводы

Ответьте на следующие вопросы в ячейке ниже (текст Markdown):

**1. Когда DBSCAN выигрывает у K-Means?**

> На нелинейных данных

**2. Когда K-Means предпочтительнее?**

> На больших данных либо когда точки имеют выпуклую округлую форму.

**3. Почему масштабирование обязательно для DBSCAN?**

> Тк DBSCAN считает евклидово расстояние, еслине масштабировать, то расстояние будет определяться признаком с большей шкалой.

**4. Как интерпретировать шумовые точки в датасете Wine?**

> Редкие сорта вин.

**5. Можно ли корректно сравнивать силуэт DBSCAN и K-Means напрямую? Почему?**

> Нельзя, тк силуэт может завышать скор для kmeans на нелинейных данных, ведь он штрафует за нелинейную форму, которую не может определять kmeans. для DBSCAN выбрасываются шумовые точки, из-за чего наборы данных уже становятся разными.